In [ ]:
from sklearn.pipeline import Pipeline

from transformers import (
    Resampler,
    BandPassFilter,
    WaveletDenoiser,
    MelSpectrogramTransformer,
    SpectrogramPadder,
    FlattenTransformer
)

from sklearn.ensemble import RandomForestClassifier

### 1. Usamos **dos pipelines separadas**

#### **A. Pipeline de entrenamiento**

Se usa para `fit()` y `predict()` sobre audios ya segmentados.
Esta es la que entrenamos y la que luego se guarda con `joblib`:

In [ ]:
pipeline_modelo = Pipeline([
    ('resampler', Resampler()),
    ('bandpass', BandPassFilter()),
    ('melspec', MelSpectrogramTransformer()),
    ('pad', SpectrogramPadder()),
    ('flatten', FlattenTransformer()),
    ('modelo', RandomForestClassifier())
])

#### **B. Pipeline de inferencia**

Se usa solo en producción.
Su tarea es: recibir un audio crudo, segmentarlo y luego aplicar la pipeline anterior **a cada ciclo**.

In [ ]:
class PipelineInferencia:
    def __init__(self, segmentador, pipeline_modelo):
        self.segmentador = segmentador
        self.pipeline_modelo = pipeline_modelo  # ya entrenada

    def predict(self, path):
        # Segmentar
        ciclos = self.segmentador.segmentar(path)
        
        predicciones = []
        for y_ciclo in ciclos:
            # Aplicar la pipeline del modelo
            X_feat = self.pipeline_modelo[:-1].transform([y_ciclo])
            pred = self.pipeline_modelo[-1].predict(X_feat)[0]
            predicciones.append(pred)
        
        return predicciones


Así, en la app solo tendrían que hacer:

In [ ]:
preds = pipeline_inferencia.predict("audio_usuario.wav")


Y `preds` sería una lista de predicciones (una por ciclo).


## 2. Segmentador

Solo corta el audio crudo:

In [ ]:

class Segmentador:
    def __init__(self, lowcut=1000, highcut=4000, ...):
        self.lowcut = lowcut
        self.highcut = highcut
        # otros hiperparámetros internos

    def segmentar(self, path):
        y, sr = librosa.load(path, sr=None)
        dur = len(y) / sr

        # segmentación con tus funciones internas
        segments = segmentar_energy_based(y, sr, ...)
        gaps = detectar_gaps(segments, dur, ...)
        ...
        ciclos = unir_en_ciclos(segments)
        ...
        ciclos = [c for c in ciclos if (c[1]-c[0]) < 6.0]

        # devolver los arrays crudos (sin modificar sample rate ni amplitud)
        return [y[int(start*sr):int(end*sr)] for start, end in ciclos]


## 3. En producción

El flujo completo queda así:

```css
audio.wav
  │
  ▼
[Segmentador] — detecta ciclos y recorta (sin alterar señal)
  │
  ├─> ciclo 1 ┐
  ├─> ciclo 2 ├─> [pipeline_modelo.transform + predict]
  └─> ciclo n ┘
  │
  ▼
  salida: lista de predicciones (una por ciclo)
```